<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/logo.png?raw=true" alt="2025年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

大規模なゲノムデータを用いた解析: ゲノムワイド関連解析(GWAS)
---

　今回から、ゲノム情報解析の一例として、大規模なゲノムデータセットを用いた解析を勉強・体験していきます。

大規模ゲノムデータ解析シリーズ:
- GWAS: Genome wide association study(GWAS)とは、GWAS(QTL解析)に必要な統計モデルの知識...等
- Genomic Prediction入門 : Genomic Predictionとは, 機械学習の基本, 統計モデル, 育種への応用...等々

また、今回(GWAS)のテキストのみ**PythonではなくR**を用いて行います。このノートのコードセルではRが動きます。

今回扱うGWASや統計解析に関してはPythonよりRの方がパッケージ等充実しているためです。(次回以降はPythonに戻ります。)

```
今回Rを使用する様に、ゲノム解析を行う上で使用する可能性があるプログラム等の知識は多岐に渡ります。
ゲノム解析と一言で言っても目的に応じてGUI(コマンドではなくクリック等で動かすアプリ)で完結する解析もあれば、自分でコードを書く必要がある解析もあります。
そのため、Pythonをやっておけば全てOKなんてことは無く、必要に応じてRを使用したり、Shell scriptを書いたり、場合によってはExcelでデータを処理することも当然あります。
また、この分野は進歩が速く、今使われている知識がずっと使用されるかと言うとそうとも限りません。3年後にはPythonに代わる別の言語がメジャーな言語になっていてPythonが一切使われなくなっている可能性もあります。
重要なのはその都度必要な知識を取り入れアップデートしていくことです。
(当然これはゲノム解析の分野だけではなく他の分野にも当てはまることです。)
```

# 今回の勉強内容
　このテキストでは、大規模ゲノムデータ解析シリーズの第1回として、GWASについて勉強します。

　内容は以下の通りです。

1. GWASとは？
  * 遺伝的変異と形質の関係
  * GWASの概要
  * GWASの解析手順
2. 必要となるデータセットの準備
  * 遺伝子型データと形質データ
3. GWASを行うための手法
  * 単純な比較（t検定）でGWASをやってみる
  * 単純な比較の限界
  * 代表的な統計モデルの紹介
  * イネを用いた実例
  * 単純なモデルと混合モデルを比べる（集団構造）
  * サンプル数やSNP数の影響
  * 多重検定: 偶然でもp値は小さくなる
  * 遺伝率: 遺伝で説明できる割合
4. GWASが行えるソフトウェアの紹介
5. (参考) ロジスティック回帰モデル

　今回は、Slido（スライド）を使ったアンケートが何回かあります。🗳 Slido の印があるところでは、先にSlidoで回答してから、次のコードセルを実行してください。

---

　一度次のコードセルを実行して、実習に必要なライブラリのインストールや、プログラム、サンプルファイルのダウンロード、この実習で使う関数の準備をしてください。

In [ ]:
################################
##  以下の実習の前に一度実行してください。##
################################
## Rパッケージのインストール、プログラム・サンプルファイルのダウンロード
system("wget -q -O library.tar.gz https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2023/dataset/library.tar.gz")
system("wget -q -O GWAS_sample_genotype_rice.csv https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2023/dataset/GWAS_sample_genotype_rice.csv")
system("wget -q -O GWAS_sample_phenotype_rice.csv https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2023/dataset/GWAS_sample_phenotype_rice.csv")
system("wget -q -O GWAS_sample_genotype.csv https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2023/dataset/GWAS_sample_genotype.csv")
system("wget -q -O GWAS_sample_phenotype.csv https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2023/dataset/GWAS_sample_phenotype.csv")
system("wget -q -O GWAS_sample.csv https://github.com/CropEvol/lecture/raw/refs/heads/master/textbook_2024/dataset/GWAS_sample.csv")
untar("library.tar.gz")
.libPaths("library")
library(rrBLUP)
library(ggplot2)

## この実習で使う関数の準備（中身を理解する必要はありません）

# rrBLUP の GWAS 関数に渡す形（marker, chrom, pos, 遺伝子型を -1/0/1 にした列）に変換する
rrBLUP_geno <- function(genotype) {
  data.frame(marker = genotype[, 1], chrom = genotype[, 2], pos = genotype[, 3],
             genotype[, 4:ncol(genotype)] - 1, check.names = FALSE)
}

# 単純な回帰分析によるGWAS（遺伝子型が2種類なら t検定と同じ）。covariates に主成分などを加えられる
gwas_simple <- function(genotype, y, covariates = NULL) {
  X <- t(as.matrix(genotype[, 4:ncol(genotype)]))        # 個体 × SNP の行列
  Q <- qr.Q(qr(cbind(rep(1, length(y)), covariates)))    # 切片と共変量
  ry <- as.vector(y - Q %*% crossprod(Q, y))              # 共変量の影響を除いた形質値
  rX <- X - Q %*% crossprod(Q, X)                         # 共変量の影響を除いた遺伝子型
  sxx <- colSums(rX^2); sxy <- colSums(rX * ry)
  beta <- sxy / sxx
  df <- length(y) - ncol(Q) - 1
  t_stat <- beta / sqrt((sum(ry^2) - beta * sxy) / df / sxx)
  score <- -(log(2) + pt(abs(t_stat), df, lower.tail = FALSE, log.p = TRUE)) / log(10)
  score[!is.finite(score)] <- 0                           # 遺伝子型が1種類しかないSNPは 0 にする
  data.frame(marker = genotype[, 1], chrom = genotype[, 2], pos = genotype[, 3],
             beta = beta, p_value = 10^(-score), score = score)
}

# Benjamini-Hochberg法で、FDRが q 未満になる -log10(p) の境界値を求める（なければ NA）
fdr_threshold <- function(score, q = 0.01) {
  q_values <- p.adjust(10^(-score), method = "BH")
  if (any(q_values < q)) min(score[q_values < q]) else NA
}

# ゲノミックコントロール値 λ（1より大きいほど、全体的にp値が小さく出すぎている）
lambda_gc <- function(score) {
  p <- 10^(-score); p <- p[p > 0 & p < 1]
  median(qchisq(p, 1, lower.tail = FALSE)) / qchisq(0.5, 1)
}

# マンハッタンプロット
manhattan <- function(df, y, threshold = NA, title = "", ylim = NULL, ylab = "-log10(p)") {
  df$index <- seq_len(nrow(df))
  g <- ggplot(df, aes(x = index, y = .data[[y]], color = chrom)) + geom_point(size = 1) +
    labs(x = "Marker", y = ylab, title = title)
  if (!is.na(threshold)) g <- g + geom_hline(yintercept = threshold, linetype = "dashed")
  if (!is.null(ylim)) g <- g + ylim(ylim)
  plot(g)
}

# QQプロット（期待される -log10(p) と、実際の -log10(p) の比較）
qqplot_gwas <- function(scores, labels, title = "") {
  d <- do.call(rbind, lapply(seq_along(scores), function(k) {
    s <- sort(scores[[k]][scores[[k]] > 0], decreasing = TRUE)
    data.frame(expected = -log10((seq_along(s) - 0.5) / length(s)), observed = s,
               model = sprintf("%s (lambda = %.2f)", labels[k], lambda_gc(s)))
  }))
  g <- ggplot(d, aes(x = expected, y = observed, color = model)) + geom_point(size = 1) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
    labs(x = "Expected -log10(p)", y = "Observed -log10(p)", title = title)
  plot(g)
}

print("準備ができました")

# 1. GWASとは？

## 1-1. 遺伝的変異と形質の関係
　GWAS(Genome-wide association study, ゲノムワイド関連解析)とは、多数の品種や系統を使い、表現型の多様性に影響を与える遺伝的変異を統計的に探索する事を指します。

　と言ってもピンとこないと思いますが、これまでに学習してきたQTL-seq等と似たような手法になるので、まずは復習がてら前回やったQTL-seqを元に説明していきます。

　前回はQTL-seqという、**遺伝子マッピング**（**Gene mapping**）手法について少し学びました。

遺伝子マッピングがどのような手法だったかというと、個体AとBのあいだの形質の違いを作っている遺伝子（原因遺伝子）がゲノム上のどこにあるのかを特定する手法です。

1. 個体AとBのゲノム配列を比較すると、個体間の塩基の違い（例えば、SNP; Single nucleotide polymorphism）を発見できます。しかし、個体AとBの間の比較だけでは、どの変異が個体間の形質の違いを作っているのかを特定できません。
2. そこで、個体AとBを交配し、ゲノムをシャッフルした個体集団（分離集団）を作ります。
3. 分離集団の各個体について、形質とゲノム上の各マーカーの遺伝子型も調べ、**形質と遺伝子型に連関**（**Association**）があるゲノム上の場所を探し、原因遺伝子を含むゲノム領域を特定します。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/gene_mapping.png?raw=true" alt="gene_mapping" height="280px" align="middle">

　上述のように、**分離集団**のマーカー遺伝子型を使って遺伝子マッピングする方法を**QTL解析**（**QTL analysis**）と言いました。QTLとはQuantitative trait locus(量的形質遺伝子座)の略称です。

　QTL-seqは、QTL解析をより省力的にした方法で、次世代シーケンスデータを用いて、分離集団の一部の個体のシーケンス情報から遺伝子マッピングをおこなう手法でした。

### (参考) QTL-seqの概要

　QTL-seqでは、分離集団のうち極端な形質値をもつ個体のバルクシーケンスデータを用いて遺伝子マッピングをおこないます。バルクシーケンス中に含まれるアリルの割合（SNP-index）の推移をゲノム全体にわたって調べることで、遺伝子の位置を特定します。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/qtlseq_overview.png?raw=true" alt="QTL-seq" height="300px" align="middle">

> <small>Takagi, H., Abe, A., Yoshida, K., Kosugi, S., Natsume, S., Mitsuoka, C., Uemura, A., Utsushi, H., Tamiru, M., Takuno, S., Innan, H., Cano, L., Kamoun, S. & Terauchi, R. (2013). [QTL‐seq: rapid mapping of quantitative trait loci in rice by whole genome resequencing of DNA from two bulked populations](https://onlinelibrary.wiley.com/doi/full/10.1111/tpj.12105). The Plant Journal, 74(1), 174-183.</small>

### GWAS

　QTL解析では、親間の形質の違いの原因となっている遺伝子の位置を探索するために、

交配による**組換え**をうまく活用して、F2集団等の分離集団からQTLの位置を検出しました。

　GWASは、分離集団に限らず、多数の品種を対象に、集団内でばらついている形質情報と密なSNPマーカー等の遺伝子型情報を得て、

形質と遺伝子型との統計的な相関関係を網羅的に探索する手法になります。

QTL解析のより網羅的・汎用的な解析と言えるかもしれません。

　F2集団では親間の変異が1度の組換えによって混ざるだけですが、自然集団では交配や組換えが繰り返し起きており、突然変異等も生じています。そのため、より多様な遺伝的変異やその変異に伴う形質の多様性が生じている可能性があります。

　そのため、GWASでは、2親間の変異だけではなく、使用した品種・系統の持つ様々な変異から形質に関与している変異を検出することになります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/QTL_vs_GWAS1.png?raw=true" alt="QTL_vs_GWAS1" height="350px" align="middle">

　このように比較してみると、GWASの方が様々なQTLが検出出来てメリットが多い様に思いますが、デメリットも当然あります。

　QTL解析では、A種とB種間の遺伝子型の違いだけに着目すれば良かったのですが、

GWASを行うためにはA種vsB種、B種vsC種、C種vsD種...といった様々な種間の遺伝子型の違いを捉える必要があります(下図のSNP_X)。

　それだけ変異のバリエーションが多いという事になるので、QTL解析ではざっくりとした領域毎のSNP情報で問題無かった所、GWASではシーケンスをしっかり読んで十分に密なSNPの情報が必要になります。

　特に、特定の１品種しか持たない遺伝的な変異の効果を捉える場合、分離集団を用いたQTL解析の方が適している可能性もあります。

　下図のSNP_Yの効果を調べるにはB種と他多数の種を比較するより、A種とB種を交配した集団で比較した方が確実です。

　分離集団を作ることで、SNP_YがAの個体・Gの個体がそれぞれ一定数出現する形になり、効果を確認しやすくなります。

　よって、QTL解析・GWASそれぞれにメリット・デメリットがあり適した使い分けが重要になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/QTL_vs_GWAS2.png?raw=true" alt="QTL_vs_GWAS2" height="350px" align="middle">

### 1-2. もう少し具体的に
　GWASでは、上述した様に、多数の品種や系統(場合によっては分離集団も)等の集団を用いて、集団の持つ遺伝的変異それぞれについて、特定の形質に対する影響の強さを考えていきます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/test_prep.png?raw=true" alt="gwas_basic1" height="200px" align="middle">

例えば、下の様な遺伝子型の情報と、葉の長さのデータが得られたとしましょう。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/GWAS_basic1.png?raw=true" alt="gwas_basic1" height="300px" align="middle">

この様な情報から、どのSNPの遺伝子型が葉の長さに影響を与えていそうかを調べていきます。

↓のファイルには10個のSNPの遺伝子型と葉の長さのデータがあります。

[GWAS_sample.csv](https://github.com/CropEvol/lecture/raw/refs/heads/master/textbook_2024/dataset/GWAS_sample.csv)


In [ ]:
# データの読み込み
data <- read.csv("GWAS_sample.csv", fileEncoding = "UTF-8-BOM")
data


　まずはこのデータから1つ目のSNPが葉の長さに影響を与えているかどうか、を調べてみましょう。

1つ目のSNPは種によって"A(アデニン)"もしくは"C(シトシン)"の遺伝子型になっています。

もしこの場所の遺伝子型がAからCに変わると葉の長さが大きく変化するのであれば、Aになっている種とCになっている種の間に葉の長さに差があるはずです。

　では皆さんの今使える知識で1つ目のSNPがAになっている種とCになっている種の間に、葉の長さに差があるのかを確認してみましょう。

例えばSNPの遺伝子型毎の葉の長さの平均値の差を見てみると、

In [ ]:
# SNP1がA(/C)の種の葉の長さを取り出す
SNP1_A <- data$Leaf[data$SNP1 == "A"]
SNP1_C <- data$Leaf[data$SNP1 == "C"]

# 平均値の差を計算
mean(SNP1_A) - mean(SNP1_C)

となります。この差（の絶対値）が大きいほど、そのSNPの遺伝子型が葉の長さに影響を与えている可能性が高いことになります。

　ただし、平均値の差だけでは、その差が「本当の違い」なのか「たまたま」なのかを判断できません。種の数が少なかったり、葉の長さのばらつきが大きかったりすると、遺伝子型と関係がなくても、偶然である程度の差が出てしまうからです。

　そこで、**t検定** を行ってみます。t検定は、2つのグループの平均値の差を、データのばらつきと数を考慮して評価し、「本当は差がないのに、偶然これくらい以上の差が出る確率」を **p値** として求めます。p値が小さいほど、偶然では説明しにくい差、つまり遺伝子型と形質の関連を示唆する差だと考えられます（一般に、p < 0.05 を目安にすることが多いです）。

　R では、`t.test` 関数で t検定をおこなえます。ここでは、2つのグループのばらつき（分散）が等しいと仮定する t検定（スチューデントのt検定）を使うので、`var.equal = TRUE` を指定します。<small>（指定しないと、分散が違うことを考慮するウェルチのt検定になり、p値が少し変わります。）</small>下記のコードを動かすと、SNPの遺伝子型ごとの葉の長さの平均値の差と、t検定のp値が求まるので、SNPの番号の部分の数字を変えて結果がどう変わるか見てみましょう。

In [ ]:
# 検定したいSNPの名前
SNP <- "SNP1"

# SNPの遺伝子型ごとに葉の長さを取り出す
genotypes <- unique(data[[SNP]])      # このSNPにある遺伝子型の種類（最初の2つを比べる）
Leaf_1 <- data$Leaf[data[[SNP]] == genotypes[1]]
Leaf_2 <- data$Leaf[data[[SNP]] == genotypes[2]]

# 平均値の差と、t検定のp値を計算する
result <- t.test(Leaf_1, Leaf_2, var.equal = TRUE)
print(paste(SNP, ":", genotypes[1], "vs", genotypes[2]))
print(paste("  平均値の差:", round(mean(Leaf_1) - mean(Leaf_2), 2)))
print(paste("  t検定のp値:", signif(result$p.value, 3)))

<small>※ SNP9 だけは遺伝子型が3種類（C, A, T）あります。このコードでは最初の2種類（C と A）だけを比べているので、T の種は計算に入っていません。</small>

### 練習1-2
　上のコードセルの `SNP = "SNP1"` を書き換えて1つずつ調べるのは手間です。`for` 文を使うことで、同じ計算を繰り返すことが可能です。`for`文を使用してSNP1〜SNP10 の平均値の差とt検定のp値をまとめて表示してみましょう。

次のコードセルの `#` の行の下に、上のコードセルの「SNPの遺伝子型ごとに〜」以降の処理を書き込んでください（`{` と `}` の間に書きます）。

In [ ]:
# 練習1-2: SNP1〜SNP10 について、平均値の差とt検定のp値をまとめて計算する
for (i in 1:10) {
  SNP <- paste0("SNP", i)
  # ここに処理を書く

}

#### 解答例

In [ ]:
for (i in 1:10) {
  SNP <- paste0("SNP", i)
  genotypes <- unique(data[[SNP]])
  Leaf_1 <- data$Leaf[data[[SNP]] == genotypes[1]]
  Leaf_2 <- data$Leaf[data[[SNP]] == genotypes[2]]
  result <- t.test(Leaf_1, Leaf_2, var.equal = TRUE)
  print(paste(SNP, " 平均値の差:", round(mean(Leaf_1) - mean(Leaf_2), 2), " p値:", signif(result$p.value, 3)))
}

　この様な比較を全ゲノム領域のSNP1つ1つに対して行っていくことで、形質値と相関の強いゲノム領域を探索することが出来ます。

　今回の結果で言うと、8番目のSNPだけが、平均値の差が大きく（約11 cm）、p値も非常に小さく（約5 × 10<sup>-8</sup>）なりました。8番目のSNPの遺伝子型が強く葉の長さに影響を与えているようです。7番目のSNPも差が約4 cmありますが、p値は0.05より大きいので、偶然の差である可能性を否定できません。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/GWAS_basic.png?raw=true" alt="gwas_basic" height="600px" align="middle">

今回は1つの染色体にたった10個のSNPという極簡単な例を見てみましたが、

GWAS(ゲノムワイド関連解析)でやりたいことは、この様にゲノム全体の中に形質と関連しているSNP等の遺伝子型を探すことになります。

## 1-3. GWASの解析手順

それでは実際に近いより大規模なデータでGWASを行う際の具体的な手順を１つずつ見ていきます。

GWASでは、様々な異なった遺伝的背景を持つ集団の「形質値」と「遺伝子型」のデータが必要ということでした。

GWASの流れ:

1. **集団の準備**
  * 大規模な系統(場合によっては交配集団)を集めて、異なる遺伝的背景を持つ集団を準備する
2. **形質データの測定**
  * 集団を栽培し、目的とする形質値を計測する
3. **SNP遺伝子型情報の取得**
  * ゲノム配列（参照配列）を用意する
  * 集団の各個体のシーケンスデータを得る
  * 参照配列にリードをアライメントする
  * SNPを検出し、各個体のSNPの遺伝子型情報を得る。
4. **統計解析**
  * 2, 3で得られた形質値データ、遺伝子型データを下の様なデータに整備し、統計解析を行いQTLを探索する。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/GWAS_process.png?raw=true" alt="GWAS_process" height="200px" align="middle">

SNPの検出に関してはこれまでに既に扱ってきたので、今回は実際に扱うデータがどの様なものか説明し、

データが得られた後の統計解析の部分をやっていきます。

# 2. 必要となるデータセットの準備

当然ですが、どれだけ大規模で信頼性の高いデータを準備できるかが、GWASで良い結果を得られるかどうかを決めます。

## 2-1. 遺伝子型データ

　基本的に集団のSNPデータになります。

各SNP毎に、SNPの位置・各個体の遺伝子型の情報が入ったテーブルデータです。

QTL-seqの時にはバルクシーケンスを行いましたが、GWASでは各個体ごとにシーケンスを行いSNPを検出します。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2024/images/GWAS_sequence.png?raw=true" alt="GWAS_sequence" height="300px" align="middle">

　バルクシーケンスでは、2回(形質値が大きい集団と小さい集団)シーケンスを行えば良かったのですが、

今回は各個体毎にシーケンスを行う必要があるので、集団サイズが大きいほどコストがかかります。

しかし、集団サイズが小さいと変異の多様性や比較に使える品種・系統の数が少なくなり、十分な検定力を得られないという問題もあります。

　QTL-seqの時にリード数が少ないと、正しくSNP等の遺伝的変異の情報が読み切れないという話もありましたが、

どれだけ研究費がかけられるのか、データのサイズがどのくらい必要なのか、様々なことを考慮して研究計画をたてる必要があります。

### データフォーマットについて

先ほどRで分析を行った際は、GWASで扱うSNPの遺伝子型を"A, T, G, C"をそのまま扱っていました。

文字のデータのままだと統計的な解析を行うことが難しくなるので、解析しやすいように数字に変換して表す場合が多いです。

今回は、簡単にするために、あるSNPが"リファレンスゲノム(基準配列)の遺伝子型"か"そうでない遺伝子型か"という2種類のデータとして扱うことにします。

例えば2倍体の生物のあるSNPの遺伝子型がリファレンスゲノムでは"A"のところ、別の系統で"T"になっていた場合、

A, T という遺伝子型のデータを 0, 2 (ヘテロは1)の様に変換したりします。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/GWAS_genotyping.png?raw=true" alt="chr to num genotype image" height="350px" align="middle">

今回は200個体の1~10番染色体にわたって位置する1000SNPの遺伝子型の情報が手に入ったとしています。

(簡略化のためヘテロ接合型は無いものとする。)

下のコードで今回扱う遺伝子型情報がどんなデータかを確認出来ます。

In [ ]:
genotype <- read.csv("GWAS_sample_genotype.csv", row.names = 1)
genotype

## 2-2. 形質データ

目的となる形質の各個体毎の計測値データです。

ただ生き物の形質値は、遺伝子の影響だけではなく環境の影響などによってぶれる可能性が考えられます。

そこで、極力そのような影響を省くことを考え、複数年の計測データなど、反復データの平均値が使われることが多いです。

今回は先ほどSNPの遺伝子型が得られている200個体について、籾数と花の色という形質のデータが得られたとします。

下のコードで今回扱う形質情報がどんなデータかを確認出来ます。

In [ ]:
phenotype <- read.csv("GWAS_sample_phenotype.csv", header = TRUE)
phenotype

### 2-2-1. 量的形質と質的形質

少し話がそれますが、形質には**量的形質**と**質的形質**があります。

量的形質は、葉の長さや籾数など、連続的/量的に変化する形質を持つ、複数の遺伝子の効果の総和によって支配されることが多い形質です。また、環境が形質に影響を与える場合が多いです。

質的形質は、花の色や抵抗性の有無など、１つまたは２つ程度の少数の遺伝子で決定されている形質で、不連続な階級に分類できる形質です。

**量的な**形質の場合、24cmとか320粒とか、そのまま計算に用いられそうですが、**質的な**形質だと式にあてはめられません。

そのため、GWASを用いて**質的な**形質に関連のある遺伝的領域を探したい場合、ロジスティック回帰など形質に応じた回帰モデルを使用する必要があります(後述)。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/GWAS_phenotype.png?raw=true" alt="GWAS_phenotype" height="200px" align="middle">

(参考)
前回まで扱っていた**QTL-seq**は量的形質を対象にしています。非常に似た手法として、**MutMap法**という手法があり、こちらは突然変異体の原因遺伝子を特定する手法で、質的形質（変異体かどうか）に向いています。

# 3. GWASを行うための手法

　ここまでで、GWASを行うための集団のGenotypeとPhenotypeのデータが準備出来ました。

ではここからは、そのデータを元にどの様にGWASを行っていくのかという過程やその際に適用する統計モデルについて、実際に試しながらもう少し詳しく勉強していきましょう。

## 3-1. 単純な比較（t検定）でGWASをやってみる

　まずは先ほどと同様に、遺伝子型間の平均値を比較することでGWASをやってみましょう。

**あるSNPの遺伝子型がA系統の遺伝子型なのかそれ以外の系統の遺伝子型なのかによって平均値にどのくらい差があるかを、t検定で調べてみます。**

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2025/images/GWAS_intro.png?raw=true" alt="ゲノム上の位置も図示" height="600px" align="middle">

1つのSNPに着目して平均値を比較してみましょう。

`marker = 2`の部分を`marker = 25`など、今回は1000個のSNPがあるので、1~1000の数字に変えてどのSNPの遺伝子型が形質値に影響を与えていそうか探してみましょう。

今回は遺伝子型に応じた形質値のグラフを描くコードも追加しています。

In [ ]:
# 検定するSNPの番号
marker <- 2

# データのフォーマットを整える
data_for_test <- data.frame(Grain.number = phenotype$Grain.number,
                            genotype = as.character(t(genotype[genotype$Marker == marker, 4:ncol(genotype)])))

# 選んだSNPの遺伝子型が0の形質値・2の形質値
phenotype_SNP_0 <- data_for_test$Grain.number[data_for_test$genotype == 0]
phenotype_SNP_2 <- data_for_test$Grain.number[data_for_test$genotype == 2]

# 平均値の差と、t検定のp値
result <- t.test(phenotype_SNP_0, phenotype_SNP_2, var.equal = TRUE)
print(paste("Marker", marker, ": 平均値の差 =", round(mean(phenotype_SNP_0) - mean(phenotype_SNP_2), 2),
            ", p値 =", signif(result$p.value, 3)))

# ボックスプロットを描くためのコード
g <- ggplot(data_for_test, aes(x = genotype, y = Grain.number, fill = genotype))
g <- g + geom_boxplot()
g <- g + geom_jitter(width = 0.2)
plot(g)

先ほどと違って今回はSNPの数が1000個あるので、1個1個数字を変えて確認していくのは手間です。

そこで、プログラムの繰り返し（`for` 文）を使うことで、全てのSNPに対して平均値の差とt検定のp値をまとめて求めてみます。

In [ ]:
# SNP1〜1000までの各SNPごとに、平均値の差とt検定のp値を求める
mean_diffs <- c()
p_values <- c()
for (i in 1:nrow(genotype)) {
  snp_genotype <- unlist(genotype[i, 4:ncol(genotype)])
  phenotype_SNP_0 <- phenotype$Grain.number[snp_genotype == 0]
  phenotype_SNP_2 <- phenotype$Grain.number[snp_genotype == 2]
  mean_diffs <- c(mean_diffs, mean(phenotype_SNP_0) - mean(phenotype_SNP_2))
  p_values <- c(p_values, t.test(phenotype_SNP_0, phenotype_SNP_2, var.equal = TRUE)$p.value)
}
GWAS_mean_diff <- data.frame(marker = genotype$Marker, chrom = genotype$Chrom, pos = genotype$Position,
                             mean_diffs = mean_diffs, p_value = p_values)
GWAS_mean_diff

このまま1000個の値を目で見るのはしんどいので、

各SNP毎に計算された平均値の差をまとめてグラフ化して表示してみます。まずは平均値の差です。

In [ ]:
# 先ほど計算した平均値の差をグラフ化（染色体ごとに色を変える）
g <- ggplot(GWAS_mean_diff, aes(x = marker, y = mean_diffs, color = chrom))
g <- g + geom_point()
g <- g + geom_hline(yintercept = 0, linetype = "dashed")
plot(g)

X軸が各マーカー(SNP)を表しており、各染色体を色で分けています。

この様に、ゲノム全体に渡って位置するSNP毎の遺伝子型の異なる個体の平均値の差をプロットとして描写することが出来ます。

この図では、

* y軸の値（遺伝子型0の平均 − 遺伝子型2の平均）がプラス側に大きい場合は、その位置のSNPの遺伝子型が2になると形質にマイナスの効果がある
* y軸の値がマイナス側に大きい場合は、その位置のSNPの遺伝子型が2になると形質にプラスの効果がある

ということになります。

今回のプロットを見ると250, 500, 750番目のSNPマーカー付近の遺伝子型が形質値と強く関連していそうです。

実際に250番目のSNPの遺伝子型毎にボックスプロットを描いてみると、このSNPの遺伝子型によって形質値が大きく変わっているのが分かるかと思います。

In [ ]:
marker <- 250

# 可視化用のデータを作る
data_for_vis <- data.frame(Grain.number = phenotype$Grain.number,
                           genotype = as.character(t(genotype[genotype$Marker == marker, 4:ncol(genotype)])))
# ボックスプロットを描く
g <- ggplot(data_for_vis, aes(x = genotype, y = Grain.number, fill = genotype))
g <- g + geom_boxplot()
g <- g + geom_jitter(width = 0.2)
plot(g)

こうした情報を元に、検出されたSNPのピーク付近に形質に効果を与える遺伝子が存在しているのではないか…といった形で研究を進めていきます。

### t検定のp値をグラフにする

　次に、t検定のp値をグラフにしてみましょう。p値は 0.05、0.001、0.00000001… のように桁が大きく変わるので、そのままではグラフにしにくい値です。そこで、**−log<sub>10</sub>(p)** に変換して表示することが一般的です。

|p値|−log<sub>10</sub>(p)|
|:--|:--|
|0.1|1|
|0.01|2|
|0.001|3|
|10<sup>-8</sup>|8|

　−log<sub>10</sub>(p) は、p値が小さいほど（偶然では説明しにくいほど）大きな値になります。

In [ ]:
GWAS_mean_diff$score <- -log10(GWAS_mean_diff$p_value)   # p値を -log10(p) に変換する

manhattan(GWAS_mean_diff, y = "score", title = "Grain number (t-test)")

　平均値の差のグラフでは、プラス・マイナスの両方向に山がありましたが、−log<sub>10</sub>(p) では、効果の向きに関係なく、関連の強い場所が上向きの山として表示されます。250番目付近の山がとても高く、750番目付近や500番目付近にも山があることがわかります。

　このように、SNPごとの検定の統計値をゲノム上の位置に並べた図を **マンハッタン・プロット** と呼びます（3-3 で詳しく説明します）。

## 3-2. 単純な比較の限界

このように、単純な平均値の比較を繰り返すことでも形質に関係していそうな遺伝領域の候補を検出できそうです。

ただ、上述したやり方では、特に現実のデータに適用するには様々な問題点があります。
* 検定を繰り返す行為(多重検定)になっていないか？
* 複数の遺伝子が影響している場合、1つのSNPの遺伝子型の影響だけに着目する様なやり方で良いのか？
* GWASでは色んな品種・系統のデータを集めて使用する場合があるので、その場合はサンプル間の環境の違いや実験区域の違いなどを考慮できない
* (家畜等でGWASを行う場合)血縁関係などが考慮できた方が良い
* ...etc

今回行ったやり方だと、着目したSNPの遺伝子型の効果しか見ていないことになるので、他にもたくさん効果を持つSNPがあった場合、正しい結果が得られない可能性があります。

そこで、着目したSNPの遺伝子型だけでなく、その他の要素も組み込める、より汎用性の高いモデルが必要になってきます。

また、先ほど紹介した質的形質を扱う場合には、質的形質に適した回帰モデルを使用する必要があります。

ex)花の色の平均値…は計算できません。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/GWAS_phenotype.png?raw=true" alt="GWAS_phenotype" height="150px" align="middle">

そのため、実際にGWASを行う際には、平均値の差の様な単純なやり方は使用されません。

次の章からは実際に利用されている代表的な統計モデルを見ていきます。

## 3-3. 代表的な統計モデルの紹介

GWASを行うための統計モデルには様々なものが存在し、今も開発が続いています。

ここでは代表的なモデルの1つとしてMixed Linear Model(MLM, 混合モデル)を紹介します。

また、対象形質が質的形質だった場合に利用することが多いロジスティック回帰についても簡単に紹介します。

(ここでは簡単な紹介だけになるので、興味がある人は自分で勉強するor生物統計学の講義を受講してみてください。)

### 3-3-1. 混合モデル

GWASでよく使用されるモデルの1つに混合モデルがあります。特に以下の様なモデルが良く使用されます。

$$
\boldsymbol{y}=W\boldsymbol{v}+X\boldsymbol{\beta}+Z\boldsymbol{u}+\boldsymbol{\epsilon}
$$

$W\boldsymbol{v}$は水準や何らかの処理の効果を固定効果として表しており、$X\boldsymbol{\beta}$は着目したSNPの効果を固定効果として表しています。$Z\boldsymbol{u}$は全ゲノム領域のポリジーン効果を変量効果として表したものになります。

つまり、個体毎の環境の違いや、他のゲノム領域の影響を考慮した上で、それでもなお着目したSNPに効果($\hat{\beta}$)があるのかを確認することが出来るモデルになっています。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/GWAS_MLM.png?raw=true" alt="GWAS_MLM" height="450px">

この様なモデルを元に、GWASを行うソフトやプログラムがたくさん開発されています。

(参考)
* Yu et al., (2006) https://www.nature.com/articles/ng1702
* Zhang et al., (2010) https://www.nature.com/articles/ng.546

#### 固定効果と変量効果

<small>*※ 読み飛ばしてOKです。混合モデルの簡単な説明をしています。*</small>

なぜ"混合"モデルと言うのかというと、固定効果と変量効果の両者を扱うことが出来るモデルなので混合モデルと呼ばれています。

固定効果・変量効果とは何なのかというと、

* 固定効果

固定効果は水準間の比較や効果の検証を目的とする要因を固定効果として扱います。

肥料100g vs 肥料200gの効果の違いや, 投薬の有り無しによる効果の違いなどです。

今回だと、あるSNPがA系統なのかB系統なのかという違いによって生じる効果を固定効果としています。

* 変量効果

変量効果は、効果にばらつきがある、分布を持つ要因を変量効果として扱います。ランダム効果等と言われたりもします。

例えば栽培した圃場の違いの効果をモデルに組み込む場合、圃場の形質への影響はある程度ばらつくことが予想されます。

(同じ圃場Xでも、端っこの方とか真ん中あたりとかでちょっと違いますよね)。

このような場合、圃場の違いは変量効果として扱います。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/GWAS_random.png?raw=true" alt="GWAS_random" height="400px">

* 混合モデル

この様な固定効果と変量効果、両者を含めたモデルを混合モデルといいます。

モデルの各効果の推定をするには、最良線形不偏推定値(Best Linear Unbiased Estimates, BLUE)や最良線形不偏予測値(Best Linear Unbiased Predictor, BLUP)という概念、分散成分の推定に一般的に使用される最尤推定法(Maximum Likelihood, ML)や制限付き最尤法(Restricted Maximum Likelihood, REML)と呼ばれる方法の知識が必要になりますが、今回は統計学の講義では無いので割愛します。

興味がある方は参考文献の佐々木先生の本が詳しいので見てみてください。

(参考文献)
* 久保拓弥 (2012) データ解析のための統計モデリング入門 一般化線形モデル・階層ベイズモデル・MCMC (確率と情報の科学) 岩波書店
* 佐々木義之（2007）変量効果の推定とBLUP 法．京都大学学術出版会

GWASで扱うデータには固定効果や変量効果の影響が及んでいるケースが多いので、両者を考慮できるようなモデルの開発が進められてきました。

#### 混合モデルによるGWASの例

　では混合モデルを自分たちで１から作って行かないとダメかというと、そんなことは無く、混合モデルを実装したプログラムやソフトウェアがあるので、それらを利用して解析していきます。

　R では `rrBLUP` や `GWASpoly` というパッケージがこの混合モデルを実装した代表的なプログラムです。今回は `rrBLUP` の `GWAS` 関数を使ってみます。

　`GWAS` 関数の計算の流れは次のとおりです（P3D法と呼ばれる方法です。EMMAX法もほぼ同じ考え方です）。

1. 全SNPから、個体どうしの遺伝的な近さ（**遺伝的関係行列**）を計算する。これが、図の $Z\boldsymbol{u}$（ポリジーン効果）のばらつき方を決める
1. 形質のばらつきを、遺伝の効果（遺伝分散）と、それ以外（残差分散）に分ける
1. 遺伝的な近さを考慮したうえで、各SNPの効果（$\hat{\beta}$）を1つずつ検定する

　`rrBLUP` では、遺伝子型を −1, 0, 1 で表すので、0, 1, 2 の遺伝子型から1を引いて渡します。

In [ ]:
library(rrBLUP)
X <- genotype[, 4:dim(genotype)[2]]
M <- X-1
geno <- data.frame(marker=genotype$Marker, chrom=genotype$Chrom, pos=genotype$Position, M, check.names=FALSE)
pheno <- phenotype[ ,c(1,2)]
GWAS_rrBLUP <- GWAS(pheno, geno, plot=FALSE)
GWAS_rrBLUP

GWASをプログラム等で実施すると、この一番右の列の値の様に、各SNP毎に形質と関連しているか検定した統計値が得られます。

`rrBLUP`ではp値を$-log10$変換した、$-log10(p)$値が一番右の列に表示されます。

$p$値は、値が小さいほど、そのSNPの遺伝子型が形質と関連している可能性が高いことを示しています。

($p$値についても詳しくは生物統計学の講義を受講するか独学してください。)

小さいほど関連している、というとややこしいので$-log10(p)$に変換して、大きいほど形質と関連している値に変換されて表記されていることが多いです。

実際にこの$-log10(p)$値をグラフ化してみると、

In [ ]:
# calculate FDR threshold
q_values <- p.adjust(10 ** (-GWAS_rrBLUP$Grain.number), method = "BH")
FDR_threshold <- min(GWAS_rrBLUP$Grain.number[q_values < 0.01])

# draw manhattan plot
g <- ggplot(GWAS_rrBLUP, aes(x = marker, y = Grain.number, color=chrom))
g <- g + geom_point()
g <- g + geom_hline(yintercept = FDR_threshold, linetype = "dashed")
plot(g)

この様にゲノム全体に渡って位置するSNP毎の$-log10(p)$値等の統計値をプロットとして描写することになります。

このような図を**マンハッタン・プロット**(**Manhattan plot**)と呼びます。

y軸の値($-log10(p)$)が高い点程、その位置のSNPの遺伝子型と形質の関連が偶然では説明しにくい（関連の証拠が強い）ことを示しています。効果の大きさそのものではない点に注意してください（効果の大きさは `beta` 列で見ます）。

#### (参考) マンハッタンプロットの点線について

マンハッタンプロットには点線があり、この点線を超えていれば、形質に関連しているSNPと言える、という一つの基準として表示してある場合が多いです。

この点線の基準がどう決められているかというと、

GWASでは、SNPの数だけ統計的な検定(関連しているかどうか確認する手法)を繰り返している形になるので、多重検定と呼ばれる問題が生じます。

統計検定を繰り返す場合は補正などをする必要があり、FDR(False discovery rate)やBonferroni補正などの手法を使って多重検定の補正を行う必要があります。

今回のマンハッタンプロットではFDR値と呼ばれる値を破線で示しており、この破線を超える統計値を示す遺伝的領域が、形質と有意に関係していると考えられることになります。

(多重検定の問題については生物統計学の講義で学ぶはずなのでここでは細かくは説明しません。取ってない方は独学してください。)

#### (参考)連鎖について

GWASを行うことで候補となる遺伝領域のピークが検出されたと思います。

大抵、ある１つのSNPだけが高く検出されるのではなく、高いSNPがいると、必然的に周囲のSNPも高くなっていることが分かります。

これは**連鎖**によって、あるSNPの近傍に位置するSNPも一緒に遺伝されることが多いため生じています。

#### 簡単なまとめ

遺伝子型データと形質データを集めて、混合モデル等を使用しGWASを実施すると、

各SNP毎に$p$値などの形質と関連している可能性を表す統計値が得られます。

得られた統計値を用いてマンハッタンプロットを描くことで、形質と関連しているSNPの位置を検出します。

## 3-4. イネを用いた実例

これまではサンプルデータで手法等を確認していました。

このサンプルデータは、籾数には250, 500, 750番目付近のSNPが、花の色には780番目付近のSNPが関わる様に意図的に作ったデータでした。平均値の差では3か所とも目立っていましたが、混合モデルでFDRの基準値を超えたのは250番目付近と751番目（ちょうど基準値の上）だけで、500番目付近（−log10(p) は約3.6）は基準値に届きませんでした。効果の小さいQTLは、GWASでは見逃されることがあります。

ここからは実際のイネの集団データを用いて、どのような解析結果になるのか確認してみましょう。

### 3-4-1. データの読み込み

今回使うデータは、イネの"ひとめぼれ"という系統に色んな系統をかけあわせた集団です。

ひとめぼれに色んなイネの系統の遺伝子が混ざっている集団になります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/paddy_field.png?raw=true" alt="paddy_field" height="400px">

この集団の遺伝子型情報と表現型情報を読み込みましょう。

まずは遺伝子型情報です。今回の場合も簡単に、0 = ひとめぼれ型、2 = 他の系統型、1 = ヘテロを表すことにします。

In [ ]:
# Genotype情報
rice_genotype <- read.csv("GWAS_sample_genotype_rice.csv")
rice_genotype

次に、表現型値の情報です。

今回は籾数と葉の幅をデータとして取得します。

In [ ]:
rice_phenotype <- read.csv("GWAS_sample_phenotype_rice.csv")
rice_phenotype

### 3-4-2. 混合モデルによるGWAS

　先ほどと同じ様に、`rrBLUP` を使って混合モデルによるGWASを行ってみましょう。形質データに籾数と葉の幅の2つの列があるので、2つの形質についてまとめて解析されます（少し時間がかかります）。

In [ ]:
library(rrBLUP)
X <- rice_genotype[, 4:dim(rice_genotype)[2]]
M <- X-1
geno <- data.frame(marker=rice_genotype$Marker, chrom=rice_genotype$Chrom, pos=rice_genotype$Position, M, check.names=FALSE)
GWAS_rice <- GWAS(rice_phenotype, geno, plot=FALSE)
GWAS_rice

各形質ごとに、SNP毎の$-log10(p)$値を算出しました。

### 3-4-3. マンハッタンプロットの描写

続いて、この$-log10(p)$値を基にマンハッタンプロットを描きます。

籾数に関して描くと、

In [ ]:
# 籾数: FDRの基準値と、-log10(p)が大きい上位5つのSNP
FDR_threshold_GN <- fdr_threshold(GWAS_rice$Grain.number, q = 0.01)
print(GWAS_rice[order(GWAS_rice$Grain.number, decreasing = TRUE), ][1:5, c(1, 2, 3, 5)])

manhattan(GWAS_rice, y = "Grain.number", threshold = FDR_threshold_GN, title = "Grain number", ylim = c(0, 12))

葉の幅に関して描くと、

In [ ]:
# 葉の幅: FDRの基準値と、-log10(p)が大きい上位5つのSNP
FDR_threshold_LW <- fdr_threshold(GWAS_rice$Leaf.width, q = 0.01)
print(paste("FDRの基準値:", FDR_threshold_LW))
print(GWAS_rice[order(GWAS_rice$Leaf.width, decreasing = TRUE), ][1:5, c(1, 2, 3, 4)])

manhattan(GWAS_rice, y = "Leaf.width", threshold = FDR_threshold_LW, title = "Leaf width", ylim = c(0, 12))

という形で、籾数に関しては形質に関与している領域が検出されましたが、

葉の幅に関しては形質に関与している領域が見つかりませんでした。

この原因としては

* 葉の幅は遺伝的な効果よりも環境などの影響を強く受ける形質である可能性

* 形質をコントロールしている遺伝子の数が非常にたくさんある(ポリジーン効果と呼ぶ)可能性

などが考えられます。どちらが正しいかは、この後の「3-8. 遺伝率」で考えます。

### 調べてみよう
　籾数で最も −log<sub>10</sub>(p) が大きかったSNPは、第3染色体の約1.38 Mbにありました。イネのゲノムデータベース [RAP-DB](https://rapdb.dna.affrc.go.jp/) の「Genome Browser」で、その周辺（例: `chr03:1000000..2000000`）を表示し、どんな遺伝子があるか見てみましょう。籾数（穂の大きさ）に関係しそうな遺伝子はありそうでしょうか？

<small>※ 遺伝子型データの位置がどの参照配列に基づくかによって、位置が少しずれている可能性があります。おおよその目安として調べてください。</small>

## 3-5. 単純なモデルと混合モデルを比べる（集団構造）

　葉の幅では、混合モデルで形質と関連するSNPは見つかりませんでした。では、3-1 のt検定と同じ単純な回帰分析（最初のコードセルで準備した `gwas_simple` 関数）で同じデータを解析すると、どうなるでしょうか？

> ### 🗳 Slido Q1
> **葉の幅を、混合モデルではなく単純な回帰分析でGWASすると、結果はどうなると思いますか？**
>
> - 混合モデルと同じく、何も見つからない
> - 形質と関連するSNPがいくつか見つかる
> - その他
>
> <small>下のコードセルを実行する前に回答してください。</small>

In [ ]:
## 単純な回帰分析（集団構造を考慮しない）
GWAS_rice_LW_simple <- gwas_simple(rice_genotype, rice_phenotype$Leaf.width)

threshold <- fdr_threshold(GWAS_rice_LW_simple$score, q = 0.01)
print(paste("FDRの基準値を超えたSNPの数:", ifelse(is.na(threshold), 0, sum(GWAS_rice_LW_simple$score >= threshold))))
manhattan(GWAS_rice_LW_simple, y = "score", threshold = threshold, title = "Leaf width (simple regression)", ylim = c(0, 12))

　単純な回帰分析では、基準値を超えるSNPがたくさん見つかりました。これは「混合モデルでは見逃していた本物の遺伝子」なのでしょうか？ それとも「偽物」なのでしょうか？

　これを確かめる方法の1つが **QQプロット** です。ほとんどのSNPは形質と関係がないはずなので、本来、p値の大部分は「偶然」の範囲に収まるはずです。QQプロットでは、偶然だけの場合に期待される −log<sub>10</sub>(p)（横軸）と、実際の −log<sub>10</sub>(p)（縦軸）を比べます。

- 点が点線（y = x）に沿っていれば、p値は偶然の範囲に収まっている
- 右上の一部の点だけが点線から上に離れていれば、本物の関連がある可能性
- 点の全体が点線より上にずれていれば、**p値が全体的に小さく出すぎている**（偽物が多い）

　全体のずれの大きさは、**λ（ラムダ、ゲノミックコントロール値）** で表され、1に近いほど良い状態です。

In [ ]:
qqplot_gwas(list(GWAS_rice_LW_simple$score, GWAS_rice$Leaf.width),
            c("Simple regression", "Mixed model"), title = "Leaf width")

　単純な回帰分析では、点の全体が点線より上にずれ、λも大きくなりました。p値が全体的に小さく出すぎていたことになります。基準値を超えたSNPの中には本物も含まれているかもしれません（たとえば第3染色体のピークは、籾数のピークと同じ場所にあり、混合モデルでも最も高い値でした）が、全体としては偽物が多く含まれていると考えられます。

　その原因の1つが **集団構造** です。今回の集団は、ひとめぼれに様々な系統を交配して作られているので、「同じ系統と交配した個体どうし」は遺伝的に似ています。そのような家系ごとに葉の幅が少し違うと、家系に特徴的なSNPが、原因遺伝子と関係がなくても形質と関連しているように見えてしまいます。

　集団構造を調べる方法として、**主成分分析（PCA）** があります。R の `prcomp` 関数で、遺伝子型から個体の主成分を計算してみましょう。

In [ ]:
## 遺伝子型（個体 × SNP）を、SNPごとに標準化してから主成分分析をおこなう
X <- t(as.matrix(rice_genotype[, 4:ncol(rice_genotype)]))
freq <- colMeans(X) / 2
use <- freq > 0 & freq < 1                             # 遺伝子型が1種類しかないSNPは除く
Z <- scale(X[, use], center = 2 * freq[use], scale = sqrt(2 * freq[use] * (1 - freq[use])))
pca <- prcomp(Z, center = FALSE)
pcs <- pca$x[, 1:10]                                   # 各個体の主成分（PC1〜PC10）
eigenvalues <- (pca$sdev^2 * (nrow(Z) - 1) / ncol(Z))[1:10]

## 左: PC1とPC2の散布図（色は葉の幅）、右: 各主成分の固有値
pca_df <- data.frame(PC1 = pcs[, 1], PC2 = pcs[, 2], Leaf.width = rice_phenotype$Leaf.width)
plot(ggplot(pca_df, aes(x = PC1, y = PC2, color = Leaf.width)) + geom_point(size = 2) + scale_color_viridis_c())
plot(ggplot(data.frame(PC = 1:10, eigenvalue = eigenvalues), aes(x = PC, y = eigenvalue)) +
       geom_col() + scale_x_continuous(breaks = 1:10) + labs(x = "Principal component", y = "Eigenvalue"))

　1つ目の図では、個体がいくつかのまとまりに分かれているでしょうか。2つ目の図は、各主成分が集団の遺伝的なばらつきをどのくらい説明しているか（固有値）を表しています。今回の集団では、固有値が少しずつしか小さくならず、少数の主成分では集団構造を表しきれていないことがわかります。たくさんの家系が混ざった集団では、このような形になります。

### 練習3-5
　主成分を共変量（説明変数）として回帰分析に加えると、集団構造の影響をある程度取り除けます。次のコードセルの `n_pcs` を 0 → 3 → 10 と変えて実行し、λ がどう変わるか確認してください。

In [ ]:
n_pcs <- 3   # 共変量に加える主成分の数（0〜10）

if (n_pcs == 0) covariates <- NULL else covariates <- pcs[, 1:n_pcs]
GWAS_rice_LW_pc <- gwas_simple(rice_genotype, rice_phenotype$Leaf.width, covariates = covariates)
print(sprintf("主成分 %d 個: lambda = %.2f", n_pcs, lambda_gc(GWAS_rice_LW_pc$score)))
print(sprintf("混合モデル  : lambda = %.2f", lambda_gc(GWAS_rice$Leaf.width)))

　主成分を加えると λ は小さくなりますが、10個加えても1付近までは下がりません。一方、混合モデルでは λ が1以下になりました。混合モデルは、遺伝的関係行列によって「すべての個体どうしの遺伝的な近さ」を考慮するので、たくさんの家系が混ざった集団の構造も扱えます。これが、GWASで混合モデルがよく使われる理由です。

<small>※ λが1より小さいのは、混合モデルがやや保守的に（p値を大きめに）出しているためと考えられます。</small>

## 3-6. サンプル数やSNP数の影響

　次に、先ほどのイネの交配集団をベースに、集団サイズが小さくなったり、検出されたSNP数が少なくなった場合に、結果がどのように変化するのか確認してみましょう。

`sample_size = 141` の部分（集団サイズ）や `SNP_size = 25781` の部分（SNPの数）の数字を色々変えてみてください。個体やSNPはランダムに選ばれるので、同じ数でも実行するたびに結果が少し変わります（1回の実行に10秒ほどかかります）。

In [ ]:
sample_size <- 141   # 10〜141
SNP_size <- 25781    # 10〜25781

# 個体とSNPをランダムに選ぶ
sample_index <- sort(sample(1:nrow(rice_phenotype), sample_size))
SNP_index <- sort(sample(1:nrow(rice_genotype), SNP_size))
tmp_genotype <- rice_genotype[SNP_index, c(1:3, 3 + sample_index)]
tmp_phenotype <- rice_phenotype[sample_index, c("id", "Grain.number")]

# GWAS
GWAS_tmp <- GWAS(tmp_phenotype, rrBLUP_geno(tmp_genotype), plot = FALSE)
threshold <- fdr_threshold(GWAS_tmp$Grain.number, q = 0.01)
print(paste("FDRの基準値:", threshold))
print(GWAS_tmp[order(GWAS_tmp$Grain.number, decreasing = TRUE), ][1:5, ])

manhattan(GWAS_tmp, y = "Grain.number", threshold = threshold,
          title = paste0("Grain number (n = ", sample_size, ", SNPs = ", SNP_size, ")"))

このように、集団サイズが十分でないと検出力が下がったり、シーケンスが十分に出来なくて検出されたSNPの数が少ないと、候補となる領域が広くなってしまいます。

その結果、GWAS自体は行うことが出来ますが、本来形質に影響を与えている領域が検出されなかったり、意味のある結果を見逃してしまう可能性もあります。

> ### 🗳 Slido Q2
> **SNPの数は 25781 のまま、集団サイズ（sample_size）を減らしていくと、第3染色体のピークが FDRの基準値を超えなくなるのは、何個体くらいからでしたか？**
>
> <small>何回か試して、ピークが基準値を超えなくなった個体数を数字で回答してください。全員の回答の分布を見てみましょう。</small>

## 3-7. 多重検定: 偶然でもp値は小さくなる

　3-3 の「(参考) マンハッタンプロットの点線について」で、多重検定の補正が必要だという話をしました。実際に、補正しないとどうなるのかを体験してみましょう。

> ### 🗳 Slido Q3
> **形質と遺伝子型がまったく関係ないデータで、25,781個のSNPをGWASしたとします。p < 0.05 になるSNPは、およそいくつあると思いますか？**
>
> <small>数字で回答してください。</small>

　次のコードセルでは、籾数の値を個体の間で**ランダムに並べ替えて**、形質と遺伝子型の関係をわざと壊したデータを作り、GWASを行います。このデータには、形質と本当に関係するSNPは1つもないはずです。

In [ ]:
pheno_null <- data.frame(id = rice_phenotype$id,
                         Grain.number = sample(rice_phenotype$Grain.number))   # 籾数をランダムに並べ替える
GWAS_null <- GWAS(pheno_null, rrBLUP_geno(rice_genotype), plot = FALSE)

n_snps <- sum(GWAS_null$Grain.number > 0)
n_p005 <- sum(GWAS_null$Grain.number > -log10(0.05))
print(paste("検定したSNPの数:", n_snps))
print(sprintf("p < 0.05 のSNP : %d 個（%.1f%%）", n_p005, n_p005 / n_snps * 100))
print(sprintf("Bonferroni補正の基準（0.05 / SNP数）: -log10(p) = %.2f", -log10(0.05 / n_snps)))
print(paste("FDR（q < 0.01）の基準値:", fdr_threshold(GWAS_null$Grain.number, q = 0.01)))

manhattan(GWAS_null, y = "Grain.number", threshold = -log10(0.05),
          title = "Shuffled grain number (dashed line: p = 0.05)", ylim = c(0, 12))

　形質と関係のないデータでも、検定したSNPの数%（3〜5%程度）で p < 0.05 になりました。p < 0.05 とは「偶然でも20回に1回くらいは起こる」という意味なので、2万回以上検定すれば、偶然だけで千個近くのSNPが「有意」に見えてしまいます。

　そこで、検定の回数に応じて基準を厳しくするのが**多重検定の補正**です。
- **Bonferroni補正**: 基準のp値を、検定の回数で割る（0.05 / 25,781 ≒ 2 × 10<sup>-6</sup>）。とても厳しい
- **FDR（偽発見率）**: 「有意」としたSNPの中に含まれる偽物の割合を、一定以下（例: 1%）に抑える。Bonferroniより緩やか

　並べ替えたデータでは、FDRの基準値が求まらない（`NA`、基準を満たすSNPがない）か、求まっても非常に高くなったはずです。

### 練習3-7
　上のコードセルを何回か実行してください。並べ替えは毎回ランダムなので、偶然一番高くなるSNPの場所は毎回変わります。−log<sub>10</sub>(p) の最大値は、毎回どのくらいになりましたか？ この値と、3-4 の籾数のピークの高さ（約10.8）を比べてみましょう。

## 3-8. 遺伝率: 遺伝で説明できる割合

　3-4 で、葉の幅から形質と関連するSNPが見つからなかった理由として、「環境の影響が強い」と「関わる遺伝子の数が非常に多い」の2つの可能性を挙げました。どちらなのかを考える手がかりが **遺伝率（heritability, h²）** です。

　遺伝率は、形質の個体間のばらつき（分散）のうち、遺伝的な違いで説明できる割合です。混合モデルでは、形質の分散を「遺伝分散」と「残差分散」に分けて推定しているので、そこから遺伝率を計算できます。`rrBLUP` の `mixed.solve` 関数を使うと、遺伝分散（`Vu`）と残差分散（`Ve`）を推定できます。

> ### 🗳 Slido Q4
> **籾数と葉の幅では、どちらの遺伝率が高いと思いますか？**
>
> - 籾数
> - 葉の幅
> - 同じくらい
>
> <small>下のコードセルを実行する前に回答してください。</small>

In [ ]:
## 遺伝的関係行列（個体どうしの遺伝的な近さ）を計算する
K <- A.mat(t(as.matrix(rice_genotype[, 4:ncol(rice_genotype)])) - 1)

## 混合モデルで遺伝分散（Vu）と残差分散（Ve）を推定し、遺伝率 = Vu / (Vu + Ve) を計算する
fit_GN <- mixed.solve(rice_phenotype$Grain.number, K = K)
fit_LW <- mixed.solve(rice_phenotype$Leaf.width, K = K)
print(sprintf("籾数の遺伝率  : %.2f", fit_GN$Vu / (fit_GN$Vu + fit_GN$Ve)))
print(sprintf("葉の幅の遺伝率: %.2f", fit_LW$Vu / (fit_LW$Vu + fit_LW$Ve)))

　葉の幅の遺伝率は、籾数と同じくらい（むしろ少し高い）でした。つまり、葉の幅も遺伝の影響を十分に受けています。それにもかかわらず、効果の大きなSNPが見つからなかったということは、葉の幅は「1つ1つの効果は小さい遺伝子が、たくさん集まって」決まっている形質（**ポリジーン**の形質）である可能性が高いと考えられます。

　このような形質では、GWASで個々の遺伝子を見つけるのは難しくなります。そこで、多数のSNPの効果をまとめて使って、形質値そのものを予測する方法が考えられました。それが次回の **Genomic Prediction（ゲノミック予測）** です。

## 4. GWASが行えるソフトウェアの紹介

GWASが行えるソフトウェアやパッケージ、ライブラリ等は無数にあります。

それぞれ異なったモデルや計算手法、考慮できる要素等があり、目的に応じて最適なものを使用する必要があります。

当然考慮したい特殊な処理区等があれば自作のプログラムで解析するのも選択肢の一つです。

ソフトウェア
* TASSEL ... https://www.maizegenetics.net/tassel
* PLINK ... https://www.cog-genomics.org/plink
* GEMMA ... https://github.com/genetics-statistics/GEMMA

Rパッケージ
* rrBLUP ... https://potatobreeding.cals.wisc.edu/software/
* GWASpoly ... https://github.com/jendelman/GWASpoly
* r/qtl ... https://rqtl.org/

Pythonライブラリ
* Hail ... https://hail.is/ （線形回帰・ロジスティック回帰・主成分分析など。混合モデルは 0.2.94 で削除）
* FaST-LMM ... https://github.com/fastlmm/FaST-LMM （混合モデル）

等々、探せばもっとたくさんあります。

## 5. (参考)ロジスティック回帰モデル

<small>*※ ここもややこしいので読み飛ばしてOKです。質的形質を扱う場合の簡単な説明をしています。*</small>

先ほど形質には**量的形質**と**質的形質**があるという話をしました。

ここからは形質が**質的な形質**だった場合の話をしていきます。

量的形質であれば、$21.4cm(葉の長さ)$や$12本(穂の数)$の様な形で数値として扱うことが出来ます。

$$
\boldsymbol{葉の長さ}=W\boldsymbol{v}+X\boldsymbol{\beta}+Z\boldsymbol{u}+\boldsymbol{\epsilon}
$$

しかし、質的形質は、$赤い(花の色)$といった形で数値ではない場合が多く、そのままでは計算に用いられません。

$$
\boldsymbol{花の色？}=W\boldsymbol{v}+X\boldsymbol{\beta}+Z\boldsymbol{u}+\boldsymbol{\epsilon}
$$

そのため、通常のモデルでは扱うことが難しくなります。ではどうするのかというと、

「白い花か赤い花か」や「抵抗性か感受性か」の様な質的形質は、あるクラスに分類される確率を求める形に変換します。

例えば、「SNP1の遺伝子型がTの時、赤い花になる確率はXX%」の様な形です。こうすると質的形質を数値として扱えるので

$\boldsymbol{赤い花になる確率}=W\boldsymbol{v}+X\boldsymbol{\beta}+Z\boldsymbol{u}+\boldsymbol{\epsilon}$という感じで計算出来そうです。

しかし、このまま量的形質と同じ様に解析をしていけばいいのかと言うとまだ注意点があります。

"確率"は扱いが難しい値で、

例えばSNP1の遺伝子型で花の色が変わる確率を対象に普通の回帰分析をしてみると、下図の様な問題が生じるので使用できません。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/logistic_limit.png?raw=true" alt="logistic_regression" height="400px">

　そこで、線形回帰モデルの代わりにロジスティック回帰モデルが用いられます。

**ロジスティック回帰**（**Logistic regression**）は、線形回帰に少し手を加えて、**あるクラスに分類される確率を求める手法**です。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2023/images/logistic_regression.png?raw=true" alt="logistic_regression" height="400px">

　ロジスティック回帰は次の式で表されるモデルです（詳しくは「ロジスティック回帰の式を詳しく　」を参照）。

$$ z = \beta x + \epsilon $$
$$ p = \frac{1}{1+e^{-z}} $$

- 係数$\beta$をSNPの効果、説明変数$x$をSNPの遺伝子型として表しています。$\epsilon$（epsilon）は切片です。
- $p$はあるクラスに属している確率を表しています。$e$はネイピア数（=2.7182...）です。  

　このとき、$p$ は $0 < p < 1$ の範囲の値しかとりませんが、$z$ は $-\infty < z < \infty$ のどんな値でもよいので、$z$ の部分を先ほどまで扱っていた線形モデルの様に扱える様になります。

　この様にロジスティック回帰モデルをたてることで、係数$\beta$や切片$\epsilon$が求まり、質的形質に対してもGWASが行えるようになるわけです。

#### ロジスティック回帰の式を詳しく...
<small>*※ 読み飛ばしてOKです。ロジスティック回帰の式の背景を数式を使って説明しています。*</small>

　ロジスティック回帰の式を理解するためには、**オッズ**(**odds**)から始める必要があります。

　ある事象Aが起こる確率を$p$としたとき、Aが起こらない確率は$1-p$となります。このとき、Aの起こる確率とAが起こらない確率の比は、$$\frac{p}{1-p}$$となります。これがオッズと呼ばれるもので、Aが起こる「見込み」を表しています。

確率である$p$の値は0から1の範囲しかとれませんが、オッズに変換することで$0から\infty$までの範囲に、さらに対数をとることで$-\infty$から$\infty$まで値を取ることが出来ます。

先程確率は0から1までしかとれないので回帰式が作れないという話をしましたが、対数オッズを対象にすると線形回帰で扱えそうな値になりますね。

さて、ここで最終的に知りたいことは、**データから確率$p$を推測する式**です。

$$
p = (SNPの情報などの式)
$$

　オッズの対数（**対数オッズ**）をとり、関数とみなすと、$$f(p) = \log{\frac{p}{1-p}}$$となります。これは、**ロジット関数（logit）**として知られる関数です。

　ロジット関数を逆関数にすると、$f(p)$から$p$を求める関数が得られます。
\begin{align*}
f(p) &= \log{\frac{p}{1-p}} \\
e^{f(p)} &= \frac{p}{1-p} \\
e^{f(p)} - e^{f(p)} p &= p \\
p &= {\frac{e^{f(p)}}{e^{f(p)}+1}} \\ &= \frac{1}{1+e^{-f(p)}}
\end{align*}

　ここで、$f(p) = z$ とすると、ロジスティック回帰のふたつ目の式が得られます。これは、**ロジスティック関数（logistic）**として知られる関数です。$$ p = \frac{1}{1+e^{-z}} $$この関数は、入力値$z$からAが起こる確率$p$を求める関数とみなせます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2019/images/logistic_curve.png?raw=true" alt="logistic_curve" height="200px">

　ロジスティック回帰は、線形回帰の式をロジスティック関数の入力$z$につなげたものです。

これは、「データx(今回だとSNPの情報)」を「Aがおこる確率$p$」に変換する式です。

$$f(p) =  z = \beta x + \epsilon $$

$$ p = \frac{1}{1+e^{-(\beta x + \epsilon)}} $$

このあたりに関しても、簡単な説明になっているので、興味のある方は各自調べてみてください。

(参考文献)

* 久保拓弥 (2012) データ解析のための統計モデリング入門 一般化線形モデル・階層ベイズモデル・MCMC (確率と情報の科学) 岩波書店

#### ロジスティック回帰による検定の例

白い花になる個体と、赤い花をつける個体から成る集団がいたとして、この質的形質である花の色と関連している遺伝領域をGWASで探してみます。

R では、`glm` 関数で `family = binomial` を指定するとロジスティック回帰をおこなえます。ここでは、全てのSNPについて1つずつロジスティック回帰をおこない、SNPを入れないモデルと比べる尤度比検定でp値を求めます。

$$ z = \beta x + \epsilon $$
$$ p = \frac{1}{1+e^{-z}} $$

<small>※ `glm` のロジスティック回帰は、固定効果（SNPや共変量）だけのモデルです。変量効果（遺伝的関係行列）も含めたロジスティック回帰は、R の `lme4qtl` などのパッケージでおこなえます（[lme4qtlの論文](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-018-2057-x)）。</small>

In [ ]:
## 花の色を、赤 = 1、白 = 0 の数値に変換する
is_red <- as.numeric(phenotype$Flower.color == "Red")
null_model <- glm(is_red ~ 1, family = binomial)           # SNPを入れないモデル

## 全SNPについてロジスティック回帰（尤度比検定）
p_values <- c()
for (i in 1:nrow(genotype)) {
  x <- unlist(genotype[i, 4:ncol(genotype)])
  model <- suppressWarnings(glm(is_red ~ x, family = binomial))
  p_values <- c(p_values, pchisq(null_model$deviance - model$deviance, df = 1, lower.tail = FALSE))
}
GWAS_flower <- data.frame(marker = genotype$Marker, chrom = genotype$Chrom, pos = genotype$Position,
                          p_value = p_values, score = -log10(p_values))
print(GWAS_flower[order(GWAS_flower$score, decreasing = TRUE), ][1:5, ])

manhattan(GWAS_flower, y = "score", threshold = fdr_threshold(GWAS_flower$score, q = 0.01),
          title = "Flower color (logistic regression)")

今回のサンプルデータは780番目付近のSNPが花の色に強い影響を与えるようなデータになっているので、その付近に大きなピークが見られると思います。

最後に、780番目のSNPの遺伝子型と花の色のクロス集計表を作ってみましょう。

### 練習5
　次のコードセルの `marker = 780` を、ピークの近くの別のSNP（例: 776、782）に変えて、遺伝子型と花の色の対応がどう変わるか確認してください。

In [ ]:
marker <- 780

table(Flower.color = phenotype$Flower.color,
      genotype = unlist(genotype[genotype$Marker == marker, 4:ncol(genotype)]))

# まとめ

　今回、GWASの概要と解析の流れをみてきました。

GWASは関心のある形質にどのような遺伝子が影響を与えているのか調べるために、非常によく用いられています。

皆さんの中でも、ゲノム解析を主な研究手法としなくても、今後原因遺伝子の探索のきっかけとしてGWASを行う方はたくさんいると思います。


また、ポピュラーな手法なので、GWASを行うための解析プログラムやパッケージが大量に存在しています。

もちろん適切な手法やモデルをたててGWASを行うことは大切ですが、

この様な大規模なデータを用いた解析の場合は、データのサイズと質が何よりも大事です。

望ましい結果が得られない時に、手法を色々変えて無理やり検出させるような人がいますが、

「どれだけのサイズのデータがあれば自分の見たいものが検出できるのか」という点をまず第１に考える必要があるでしょう。

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>